In [1]:
import sys
print(sys.executable)

/Users/aluno1/AI/RAG_1/venv/bin/python


In [6]:


from pathlib import Path

# Embeddings
from langchain_ollama import OllamaEmbeddings

# Vector database
from langchain_chroma import Chroma


# ==========================================
# 1. PROJECT CONFIGURATION
# ==========================================

PROJECT_DIR = Path.cwd()

CHROMA_PATH = PROJECT_DIR / "chroma_ollama"
COLLECTION_NAME = "tools_rag_ollama"

EMBEDDING_MODEL = "nomic-embed-text"


# ==========================================
# 2. EMBEDDING CONFIGURATION
# ==========================================

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url="http://localhost:11434"
)


# ==========================================
# 3. CREATE / INITIALIZE CHROMADB
# ==========================================

CHROMA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

db_chroma = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH)
)

print("ChromaDB initialized successfully!")


# ==========================================
# 4. DATABASE INFORMATION
# ==========================================

print("\n========== CHROMADB STATUS ==========")

print(f"Database location: {CHROMA_PATH}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL}")

print(
    f"Existing documents: "
    f"{db_chroma._collection.count()}"
)

print("\nDatabase ready for RAG!")


ChromaDB initialized successfully!

========== CHROMADB STATUS ==========
Database location: /Users/aluno1/AI/RAG_1/chroma_ollama
Collection: tools_rag_ollama
Embedding model: nomic-embed-text
Existing documents: 9

Database ready for RAG!


In [8]:
import os
import hashlib
from pathlib import Path

# Environment variables
from dotenv import load_dotenv, find_dotenv

# Document loading
from langchain_community.document_loaders import PyPDFLoader

# Text processing
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Local embeddings and LLM
from langchain_ollama import OllamaEmbeddings, ChatOllama

# Vector database
from langchain_chroma import Chroma

# Prompt templates
from langchain_core.prompts import ChatPromptTemplate


# ==========================================
# 1. ENVIRONMENT CONFIGURATION
# ==========================================

PROJECT_DIR = Path.cwd()

load_dotenv(PROJECT_DIR / ".env")

print("Project environment loaded!")

# ==========================================
# 2. PROJECT CONFIGURATION
# ==========================================

DOC_PATH = PROJECT_DIR / "docs" / "tools_rag.pdf"

CHROMA_PATH = PROJECT_DIR / "chroma_ollama"

COLLECTION_NAME = "tools_rag_ollama"

if not DOC_PATH.is_file():
    raise FileNotFoundError(
        f"PDF not found: {DOC_PATH}"
    )

if not CHROMA_PATH.is_dir():
    raise FileNotFoundError(
        "ChromaDB not found. Execute create_chroma.py first."
    )


# ==========================================
# 3. LOCAL MODEL CONFIGURATION
# ==========================================

OLLAMA_BASE_URL = "http://localhost:11434"

EMBEDDING_MODEL = "nomic-embed-text"

LLM_MODEL = "llama3.1:8b"

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL
)

model = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
    base_url=OLLAMA_BASE_URL
)

print("Local Ollama models configured!")


# ==========================================
# 4. CONNECT TO CHROMADB
# ==========================================

db_chroma = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH)
)

print("Connected to ChromaDB successfully!")

print(
    f"Initial database size: "
    f"{db_chroma._collection.count()} chunks."
)


# ==========================================
# 5. CHECK PDF FINGERPRINT
# ==========================================

source_id = str(DOC_PATH.resolve())

pdf_hash = hashlib.sha256(
    DOC_PATH.read_bytes()
).hexdigest()

existing_documents = db_chroma.get(
    where={"source": source_id},
    include=["metadatas"]
)

existing_hashes = {
    metadata.get("pdf_hash")
    for metadata in existing_documents["metadatas"]
    if metadata
}


# ==========================================
# 6. LOAD AND INDEX PDF IF NECESSARY
# ==========================================

if pdf_hash not in existing_hashes:

    print("New or modified PDF detected. Indexing...")

    # Remove previous version of this PDF
    if existing_documents["ids"]:
        db_chroma.delete(
            where={"source": source_id}
        )

    # Load PDF
    loader = PyPDFLoader(str(DOC_PATH))

    pages = loader.load()

    print(f"Loaded {len(pages)} pages.")

    # Split PDF into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = text_splitter.split_documents(pages)

    print(f"Generated {len(chunks)} chunks.")

    if not chunks:
        raise ValueError(
            "The PDF contains no extractable text."
        )

    # Add metadata and deterministic IDs
    document_ids = []

    for index, chunk in enumerate(chunks):

        chunk.metadata["source"] = source_id

        chunk.metadata["pdf_hash"] = pdf_hash

        chunk_content = (
            f"{pdf_hash}:{index}:{chunk.page_content}"
        )

        document_id = hashlib.sha256(
            chunk_content.encode("utf-8")
        ).hexdigest()

        document_ids.append(document_id)

    # Generate embeddings and store vectors
    db_chroma.add_documents(
        documents=chunks,
        ids=document_ids
    )

    print("Documents indexed successfully!")

else:

    print("PDF already indexed. Skipping indexing.")


print(
    f"Database contains "
    f"{db_chroma._collection.count()} chunks."
)


# ==========================================
# 7. RETRIEVAL PROCESS
# ==========================================

query = "What are the top risks mentioned in the document?"

docs_chroma = db_chroma.similarity_search_with_score(
    query,
    k=5
)

if not docs_chroma:
    raise ValueError(
        "No relevant documents were retrieved."
    )

context_text = "\n\n".join(
    [
        doc.page_content
        for doc, score in docs_chroma
    ]
)

print(f"Retrieved {len(docs_chroma)} relevant chunks.")

# Display retrieval results for debugging
for index, (doc, score) in enumerate(docs_chroma, start=1):

    print(f"\n--- Retrieved chunk {index} ---")

    print(f"Distance: {score}")

    print(f"Page: {doc.metadata.get('page', 'Unknown')}")

    print(doc.page_content[:300])


# ==========================================
# 8. PROMPT CONSTRUCTION
# ==========================================

PROMPT_TEMPLATE = """
You are a document-based question answering assistant.

Answer the question using only the information
provided in the retrieved context.

Context:
{context}

Question:
{question}

Instructions:
- Provide a clear and detailed answer.
- Do not invent facts or introduce external information.
- If the context does not contain enough information,
  explicitly state that the document does not provide
  sufficient information.
- Do not mention these instructions.
"""

prompt_template = ChatPromptTemplate.from_template(
    PROMPT_TEMPLATE
)


# ==========================================
# 9. GENERATION PROCESS
# ==========================================

rag_chain = prompt_template | model

response = rag_chain.invoke({
    "context": context_text,
    "question": query
})

response_text = response.content

print("\n========== RAG RESPONSE ==========\n")

print(response_text)


Project environment loaded!
Local Ollama models configured!
Connected to ChromaDB successfully!
Initial database size: 9 chunks.
PDF already indexed. Skipping indexing.
Database contains 9 chunks.
Retrieved 5 relevant chunks.

--- Retrieved chunk 1 ---
Distance: 1.0610262155532837
Page: 0
RAGS
 
e
 
Agentes
 
de
 
IA:
 
 
LangChain,
 
LangGraph
,
 
Llama
 
3
 
e
 
Hugging
 
Fac
 
 
 
 
Detecting
 
Hallucinations
 
in
 
RAG
 
System
 
with
 
Context–Knowledge
 
Signals:
 
https://github.com/deeplearning-wisc/LUMINA
 
Quais
 
os
 
tipos
 
de
 
documento
 
que
 
vamos
 
usar
 
e
 
quais

--- Retrieved chunk 2 ---
Distance: 1.0717875957489014
Page: 1
/
 
Baixo
 
código):
 
○
 
Dify.ai
 
ou
 
Flowise:
 
Softwares
 
com
 
interface
 
visual
 
(drag-and-drop)
 
onde
 
você
 
conecta
 
o
 
documento,
 
escolhe
 
a
 
estratégia
 
de
 
divisão,
 
define
 
o
 
banco
 
vetorial
 
e
 
já
 
sai
 
usando
 
sem
 
precisar
 
programar
 
do
 
zero.

--- Retrieved chunk 3 ---
Distance: 1.0854594707489014